In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
from src.utils.utils import find_project_root 

BASE_DIR = find_project_root()

DATA_DIR = BASE_DIR / "data" / "processed"

from src.preprocessing.raw_data_preprocessing import * 

#### Load raw data (experiment 3)

In [7]:
# Load data
raw_data, meta_data, features, barcodes = load_files(exp1=False)

#### Validate and prepare single cell data

In [8]:
counts, gene_names, cell_barcodes, features, meta_data = validate_and_prepare_data(
    raw_data, meta_data, features
)

Matrix shape (cells × genes): (28611, 27999)


#### Quality control filtering steps

In [9]:
counts, meta_data, qc_mask, n_genes_per_cell, mito_pct_per_cell = apply_qc_filtering(
    counts, features, meta_data, threshold=4000
)


Step 1 — QC cell filtering (genes > 4000 & mito% < 10%)
  Removed : 3846 cells
  Retained: 24765 cells
  Matrix after cell filtering: (24765, 27999)


In [10]:
counts, meta_data, features = remove_unknown_cells(counts, meta_data, features)


Step 2 — Remove unknown cells
  Removed : 4273 cells
  Retained: 20492 cells
  Matrix after removing unknowns: (20492, 27999)


#### Gene filtering:

In [11]:
counts, features, meta_data, gene_names = filter_genes(counts, features, meta_data)



Step 3 — Gene filtering (remove zero-count genes)
  Removed : 9080 genes
  Retained: 18919 genes
  Matrix after gene filtering: (20492, 18919)


In [12]:
counts, features, gene_names = merge_duplicate_genes(counts, features, gene_names)


Step 4 — Merge duplicate gene symbols
  Genes before: 18919
  Duplicate gene names collapsed: 13
  Genes after : 18906
  Matrix after merging: (20492, 18906)


#### CPM Normalization

In [13]:
cell_barcodes = meta_data["cell_barcode"].values
cpm_counts = apply_cpm_normalization(counts, gene_names, cell_barcodes)


Step 5 — CPM normalisation
  Library size — min: 15064.0, median: 37584, max: 155157.0
  ✓ normalized_df shape: (20492, 18906)


In [14]:
#make cpm data frame:
cpm_df = pd.DataFrame(
    cpm_counts.toarray().astype(np.float32),
    index=meta_data["sample_id"],
    columns=gene_names
)
cpm_df.index.name = None

h2b_vec = cpm_df.pop("H2BCITRINE")

In [17]:
cpm_df.to_parquet(Path(DATA_DIR) / "cpm_df.parquet", compression='snappy')
h2b_vec.to_frame().to_parquet(Path(DATA_DIR) / "cpm_h2b_vec.parquet", compression='snappy')

#### Add Barcodes to index

In [3]:
cpm_df = pd.read_parquet(Path(DATA_DIR) / "cpm_df.parquet")

In [19]:
cpm_df.set_index(cpm_df.index.astype(str) + "_" + meta_data['cell_barcode'].values, inplace=True)

In [21]:
cpm_df.to_parquet(Path(DATA_DIR) / "cpm_df_with_barcodes.parquet", compression='snappy')